# sgd-vanilla-from-scratch — faded example 2: Zero gradients after the SGD step

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `sgd-vanilla-from-scratch`. Running the beacon reports progress on the `Optimizer: SGD vanilla from scratch` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: SGD vanilla from scratch` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`sgd-vanilla-from-scratch`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "sgd-vanilla-from-scratch"
DD_SUBTOPIC = "Optimizer: SGD vanilla from scratch"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

After each SGD step, gradients must be zeroed so the next `backward()` call does not accumulate on top of old gradients. Setting `p.grad = None` is the recommended pattern because it frees the gradient tensor's memory rather than writing zeros into it. The next backward call will allocate a fresh gradient tensor starting from zero.

## Faded exercise 2

Complete the `sgd_step_zero_grads(params, lr)` function. After updating each parameter, the gradient must be cleared.

1. Skip params with grad=None.
2. Apply in-place update `p.array -= lr * p.grad`.
3. Clear the gradient for the next backward.

The blank step is clearing p.grad after the update.

**Fill in:** Set p.grad to None to clear the accumulated gradient after the parameter has been updated.

In [ ]:
import torch as t

t.manual_seed(0)

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = array
        self.requires_grad = requires_grad
        self.grad = None
        self.recipe = None

def sgd_step_zero_grads(params, lr):
    for p in params:
        if p.grad is None:
            continue
        p.array -= lr * p.grad
        raise NotImplementedError()  # TODO: Set p.grad to None to clear the accumulated gradient after the parameter has been updated.

p1 = MiniTensor(t.tensor([2.0]), requires_grad=True)
p2 = MiniTensor(t.tensor([6.0]), requires_grad=True)
p1.grad = t.tensor([1.0])
p2.grad = t.tensor([3.0])

sgd_step_zero_grads([p1, p2], 0.2)
print(f'p1.array = {p1.array.item():.3f}  (expected 1.8)')
print(f'p2.array = {p2.array.item():.3f}  (expected 5.4)')
print(f'p1.grad is None: {p1.grad is None}  (should be True)')
print(f'p2.grad is None: {p2.grad is None}  (should be True)')


def _test():
    import torch as t

    class MiniTensor:
        def __init__(self, array, requires_grad=False):
            self.array = array
            self.requires_grad = requires_grad
            self.grad = None
            self.recipe = None

    p1 = MiniTensor(t.tensor([2.0]), requires_grad=True)
    p2 = MiniTensor(t.tensor([6.0]), requires_grad=True)
    p1.grad = t.tensor([1.0])
    p2.grad = t.tensor([3.0])
    sgd_step_zero_grads([p1, p2], 0.2)
    assert abs(p1.array.item() - 1.8) < 1e-6, f'p1={p1.array.item()}'
    assert abs(p2.array.item() - 5.4) < 1e-6, f'p2={p2.array.item()}'
    assert p1.grad is None, 'p1.grad should be None'
    assert p2.grad is None, 'p2.grad should be None'


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

t.manual_seed(0)

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = array
        self.requires_grad = requires_grad
        self.grad = None
        self.recipe = None

def sgd_step_zero_grads(params, lr):
    for p in params:
        if p.grad is None:
            continue
        p.array -= lr * p.grad
        p.grad = None

p1 = MiniTensor(t.tensor([2.0]), requires_grad=True)
p2 = MiniTensor(t.tensor([6.0]), requires_grad=True)
p1.grad = t.tensor([1.0])
p2.grad = t.tensor([3.0])

sgd_step_zero_grads([p1, p2], 0.2)
print(f'p1.array = {p1.array.item():.3f}  (expected 1.8)')
print(f'p2.array = {p2.array.item():.3f}  (expected 5.4)')
print(f'p1.grad is None: {p1.grad is None}  (should be True)')
print(f'p2.grad is None: {p2.grad is None}  (should be True)')
```
</details>